In [ ]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from pathlib import Path
import gc

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm
from dask.diagnostics import ProgressBar


In [ ]:
PATH = Path("/scratch/ng72/ha2606/barra_re2_wind100_monthly_nc/")
OUT_DIR = Path("/scratch/ng72/ha2606/barra_re2_wind100_thresholds/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "wind100_P20_threshold_1991_2020.nc"

monthly_files = sorted(PATH.glob("wind100_*.nc"))

print(f"Monthly files found: {len(monthly_files)}")

if len(monthly_files) == 0:
    raise FileNotFoundError(f"No NetCDF files found in: {PATH}")

In [ ]:
paths = pd.DataFrame(monthly_files, columns=["path"])

paths["year"] = (
    paths["path"]
    .astype(str)
    .str.split("_")
    .str[-1]
    .str.split("-")
    .str[0]
    .astype(int)
)

paths["month"] = (
    paths["path"]
    .astype(str)
    .str.split("_")
    .str[-1]
    .str.split("-")
    .str[-1]
    .str.split(".")
    .str[0]
    .astype(int)
)

paths = paths.sort_values(["year", "month"]).reset_index(drop=True)

print(paths.head())
print(paths.tail())

In [ ]:
CLIM_START = 1990
CLIM_END = 2025

clim_paths = paths.loc[
    (paths["year"] >= CLIM_START) &
    (paths["year"] <= CLIM_END)
].sort_values(["year", "month"]).reset_index(drop=True)

clim_files = clim_paths["path"].tolist()

print(f"Climatology files found: {len(clim_files)}")
print("First climatology file:", clim_files[0])
print("Last climatology file :", clim_files[-1])

print(clim_paths.groupby("year")["month"].count())

In [ ]:
VAR_NAME = "wind100"

def preprocess(ds):
    if VAR_NAME not in ds.data_vars:
        raise KeyError(f"{VAR_NAME} not found. Found variables: {list(ds.data_vars)}")
    return ds[[VAR_NAME]]


ds_clim = xr.open_mfdataset(
    clim_files,
    combine="by_coords",
    preprocess=preprocess,
    engine="netcdf4",
    decode_times=True,
    chunks={
        "time": 31,
        "latitude": 50,
        "longitude": 50,
    },
    parallel=False,
    data_vars="minimal",
    coords="minimal",
    compat="override",
    combine_attrs="drop_conflicts",
)

print(ds_clim)

In [ ]:
wind100 = ds_clim[VAR_NAME].astype("float32")
wind100 = wind100.sortby("time")

_, unique_idx = np.unique(wind100["time"].values, return_index=True)
wind100 = wind100.isel(time=np.sort(unique_idx))

print(wind100)
print("Time range:", str(wind100.time.values[0]), "to", str(wind100.time.values[-1]))

In [ ]:
# Important:
# Your data dimensions are: time, realization, lon, lat
# This calculates P20 separately for each realization and each grid cell.

wind100_for_q = wind100.chunk({
    "time": -1,          # full 1991-2020 time series
    "realization": 1,    # keep each realization separate
    "lon": 40,
    "lat": 40,
})

p20 = wind100_for_q.quantile(
    0.20,
    dim="time",
    skipna=True,
).astype("float32")

# Clean quantile dimension / coordinate
if "quantile" in p20.dims:
    p20 = p20.sel(quantile=0.20, drop=True)

if "quantile" in p20.coords:
    p20 = p20.drop_vars("quantile")

p20 = p20.rename("wind100_p20")

p20.attrs["long_name"] = "20th percentile wind100 threshold"
p20.attrs["description"] = (
    "Climatological 20th percentile of wind100 calculated from 1991 to 2020, "
    "separately for each realization"
)
p20.attrs["climatology_start_year"] = CLIM_START
p20.attrs["climatology_end_year"] = CLIM_END
p20.attrs["percentile"] = 20
p20.attrs["source_variable"] = VAR_NAME
p20.attrs["realization_handling"] = "P20 calculated separately for each realization"

print(p20)

In [ ]:
from pathlib import Path
import os
import gc
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from netCDF4 import Dataset

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# Close old Dask cluster if it exists
try:
    client.close()
    cluster.close()
except Exception:
    pass

VAR_NAME = "wind100"
CLIM_START = 1991
CLIM_END = 2020

PATH = Path("/scratch/ng72/ha2606/barra_re2_wind100_monthly_nc/")
OUT_DIR = Path("/scratch/ng72/ha2606/barra_re2_wind100_thresholds/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "wind100_P20_threshold_1991_2020_by_realization.nc"

# Use all 48 CPUs
N_WORKERS = 48

# Spatial tile size
# Bigger tile = fewer file opens, more memory
# With your memory, 40 x 40 is reasonable
LON_CHUNK = 40
LAT_CHUNK = 40

monthly_files = sorted(PATH.glob("wind100_*.nc"))

paths = pd.DataFrame(monthly_files, columns=["path"])

paths["year"] = (
    paths["path"].astype(str)
    .str.split("_").str[-1]
    .str.split("-").str[0]
    .astype(int)
)

paths["month"] = (
    paths["path"].astype(str)
    .str.split("_").str[-1]
    .str.split("-").str[-1]
    .str.split(".").str[0]
    .astype(int)
)

paths = paths.sort_values(["year", "month"]).reset_index(drop=True)

clim_paths = paths.loc[
    (paths["year"] >= CLIM_START) &
    (paths["year"] <= CLIM_END)
].sort_values(["year", "month"]).reset_index(drop=True)

clim_files = clim_paths["path"].tolist()

print(f"Climatology files: {len(clim_files)}")
print("First:", clim_files[0])
print("Last :", clim_files[-1])
print(clim_paths.groupby("year")["month"].count())

In [ ]:
# Read coordinates from first file
with xr.open_dataset(clim_files[0], engine="netcdf4", decode_times=True) as ds0:
    da0 = ds0[VAR_NAME]

    print("Variable dims:", da0.dims)
    print("Variable shape:", da0.shape)

    lon_vals = ds0["lon"].values
    lat_vals = ds0["lat"].values

    if "realization" in ds0.coords:
        realization_vals = ds0["realization"].values
    else:
        realization_vals = np.arange(ds0.sizes["realization"])

n_real = len(realization_vals)
n_lon = len(lon_vals)
n_lat = len(lat_vals)

print("n_realization:", n_real)
print("n_lon:", n_lon)
print("n_lat:", n_lat)

# Remove old output file
if OUT_FILE.exists():
    print(f"Removing old file: {OUT_FILE}")
    OUT_FILE.unlink()

# Create NetCDF output file
with Dataset(OUT_FILE, "w", format="NETCDF4") as nc:
    nc.createDimension("realization", n_real)
    nc.createDimension("lon", n_lon)
    nc.createDimension("lat", n_lat)

    rvar = nc.createVariable("realization", realization_vals.dtype, ("realization",))
    lonvar = nc.createVariable("lon", lon_vals.dtype, ("lon",))
    latvar = nc.createVariable("lat", lat_vals.dtype, ("lat",))

    rvar[:] = realization_vals
    lonvar[:] = lon_vals
    latvar[:] = lat_vals

    outvar = nc.createVariable(
        "wind100_p20",
        "f4",
        ("realization", "lon", "lat"),
        zlib=True,
        complevel=1,
        shuffle=True,
        fill_value=np.nan,
    )

    outvar.long_name = "20th percentile wind100 threshold"
    outvar.description = "Climatological 20th percentile of wind100 calculated from 1991 to 2020, separately for each realization"
    outvar.climatology_start_year = CLIM_START
    outvar.climatology_end_year = CLIM_END
    outvar.percentile = 20
    outvar.source_variable = VAR_NAME
    outvar.realization_handling = "P20 calculated separately for each realization"

    nc.title = "wind100 P20 threshold by realization"
    nc.source = "BARRA-RE2 wind100 monthly NetCDF files"

print(f"Created output file: {OUT_FILE}")

In [ ]:
def compute_p20_tile(args):
    """
    Compute P20 for one lon-lat tile.

    Returns:
        lon_start, lon_stop, lat_start, lat_stop, p20_tile

    p20_tile shape:
        realization, lon_tile, lat_tile
    """
    lon_start, lon_stop, lat_start, lat_stop, clim_files = args

    blocks = []

    for fp in clim_files:
        with xr.open_dataset(fp, engine="netcdf4", decode_times=True) as ds:
            da = ds[VAR_NAME].isel(
                lon=slice(lon_start, lon_stop),
                lat=slice(lat_start, lat_stop),
            )

            # Make sure dimension order is consistent
            da = da.transpose("time", "realization", "lon", "lat")

            arr = da.values.astype("float32")
            blocks.append(arr)

    # Combine all monthly data along time
    all_time = np.concatenate(blocks, axis=0)

    # P20 along time
    p20_tile = np.nanpercentile(all_time, 20, axis=0).astype("float32")

    del blocks, all_time
    gc.collect()

    return lon_start, lon_stop, lat_start, lat_stop, p20_tile


# Build tile list
tasks = []

for lon_start in range(0, n_lon, LON_CHUNK):
    lon_stop = min(lon_start + LON_CHUNK, n_lon)

    for lat_start in range(0, n_lat, LAT_CHUNK):
        lat_stop = min(lat_start + LAT_CHUNK, n_lat)

        tasks.append((lon_start, lon_stop, lat_start, lat_stop, clim_files))

print(f"Total tiles: {len(tasks)}")
print(f"Workers: {N_WORKERS}")

ctx = mp.get_context("fork")

with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as executor:

    futures = [executor.submit(compute_p20_tile, task) for task in tasks]

    # Main process writes to NetCDF safely
    with Dataset(OUT_FILE, "a", format="NETCDF4") as nc:

        outvar = nc.variables["wind100_p20"]

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Computing and writing P20 tiles",
            unit="tile",
        ):
            lon_start, lon_stop, lat_start, lat_stop, p20_tile = future.result()

            outvar[:, lon_start:lon_stop, lat_start:lat_stop] = p20_tile

            del p20_tile
            gc.collect()

print(f"Finished P20 threshold file: {OUT_FILE}")